<a href="https://colab.research.google.com/github/SarikMaj/shapely-project/blob/main/shapely_projekt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knihovna Shapely

**Semestrální projekt – představení knihovny**

Pracujeme s otevřenými daty **Natural Earth**, která pokrývají celý svět (státy, regiony, sídla, řeky). Většinu ukázek zaměříme na **Českou republiku**, protože výsledky tu snadno ověříme: spočítáme rozlohy krajů, zjistíme, které kraje spolu sousedí, ve kterém kraji leží které město, jak daleko jsou města od hranice nebo která leží u řeky. Na závěr zpracujeme sídla celého světa najednou.

## Co je Shapely?

**Shapely** je knihovna pro práci s **geometrií** – body, liniemi a polygony. Umožňuje:

* vytvářet geometrie a zjišťovat jejich vlastnosti (plocha, délka, obalový obdélník…),
* testovat **prostorové vztahy** (leží uvnitř? dotýká se? protíná?),
* provádět **geometrické operace** (průnik, sjednocení, rozdíl, buffer, zjednodušení…).

Uvnitř stojí na C++ knihovně **GEOS**, kterou používají i **QGIS** a **PostGIS** – výsledky jsou proto stejné jako v těchto nástrojích. Shapely je také základem knihovny **GeoPandas**: každý prvek ve sloupci `geometry` je objekt Shapely.

Od verze **2.0** nabízí **vektorizované funkce**, které zpracují celé pole geometrií najednou a jsou mnohem rychlejší než cykly v Pythonu.

In [ ]:
# V Google Colab je vše předinstalované, lokálně: pip install shapely geopandas pyproj matplotlib
import time
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import pyproj
import shapely
from shapely import Point, LineString, Polygon, MultiPoint
from shapely.ops import nearest_points
from shapely.plotting import plot_polygon, plot_line, plot_points

warnings.filterwarnings("ignore", category=RuntimeWarning, module="pyogrio")

print("Shapely:", shapely.__version__, "| GEOS:", shapely.geos_version_string)

## 1. Data

Použijeme otevřená data **[Natural Earth](https://www.naturalearthdata.com/)** v měřítku 1 : 10 mil. (public domain):

| Vrstva | Obsah | Typ geometrie |
|---|---|---|
| `ne_10m_admin_0_countries` | státy světa | polygony |
| `ne_10m_admin_1_states_provinces` | kraje (a regiony dalších států) | polygony |
| `ne_10m_populated_places_simple` | významná sídla světa | body |
| `ne_10m_rivers_lake_centerlines`, `ne_10m_rivers_europe` | řeky | linie |

Všechny vrstvy pokrývají celý svět. Data se při prvním spuštění automaticky stáhnou do složky `data/`. K jejich načtení použijeme GeoPandas – hned potom ale budeme pracovat s „čistými“ objekty Shapely.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/{}.geojson"

def nacti(nazev):
    """Stáhne vrstvu Natural Earth (pokud ještě není stažená) a načte ji."""
    soubor = DATA_DIR / f"{nazev}.geojson"
    if not soubor.exists():
        print(f"Stahuji {nazev} …")
        urlretrieve(URL.format(nazev), soubor)
    gdf = gpd.read_file(soubor)
    gdf.columns = gdf.columns.str.lower()
    return gdf

staty = nacti("ne_10m_admin_0_countries")
regiony = nacti("ne_10m_admin_1_states_provinces")
mesta_svet = nacti("ne_10m_populated_places_simple")
reky_svet = pd.concat([nacti("ne_10m_rivers_lake_centerlines"), nacti("ne_10m_rivers_europe")], ignore_index=True)

# z celosvětových vrstev vybereme ČR a její kraje
cr = staty.loc[staty["adm0_a3"] == "CZE", "geometry"].iloc[0]
kraje = regiony.loc[regiony["adm0_a3"] == "CZE", ["name", "geometry"]].reset_index(drop=True)
kraje["name"] = kraje["name"].replace({"Prague": "Praha"})

print(f"Států: {len(staty)}, sídel: {len(mesta_svet)}, úseků řek: {len(reky_svet)}, krajů ČR: {len(kraje)}")

### Typy geometrie na skutečných objektech

Shapely rozlišuje několik typů geometrie podle standardu *Simple Features*. Ukážeme si je přímo na našich datech:

In [ ]:
stredocesky = kraje.loc[kraje["name"] == "Středočeský", "geometry"].iloc[0]
vltava = shapely.union_all(reky_svet.loc[reky_svet["name"] == "Vltava", "geometry"].to_numpy())
praha_bod = mesta_svet.loc[mesta_svet["name"] == "Prague", "geometry"].iloc[0]

for nazev, g in [("Česká republika", cr), ("Středočeský kraj", stredocesky),
                 ("Vltava", vltava), ("Praha (sídlo)", praha_bod)]:
    print(f"{nazev:18} {g.geom_type:16} vrcholů: {shapely.get_num_coordinates(g):5}")

print("\nPočet děr ve Středočeském kraji:", len(stredocesky.interiors), "← tou dírou je Praha")
print("Praha ve formátu WKT:", praha_bod)
print("Obalový obdélník ČR:", [round(v, 2) for v in cr.bounds])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_polygon(stredocesky, ax=ax, add_points=False, color="tab:green", alpha=0.4)
plot_line(vltava.intersection(stredocesky.buffer(0.05)), ax=ax, add_points=False, color="tab:blue")
plot_points(praha_bod, ax=ax, color="tab:red", markersize=8)
ax.set_title("Polygon s dírou (Středočeský kraj), linie (Vltava) a bod (Praha)")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 2. Rozlohy krajů

Shapely počítá v rovině a v jednotkách, v jakých jsou souřadnice. Data jsou v zeměpisných stupních (WGS 84), proto je nejdřív převedeme do českého souřadnicového systému **S-JTSK / Křovák (EPSG:5514)**, který je v metrech. Převod souřadnic spočítá knihovna **pyproj**, Shapely ho funkcí `shapely.transform` aplikuje na všechny vrcholy.

In [ ]:
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:5514", always_xy=True)

def do_krovaka(geom):
    """Převede geometrii (nebo pole geometrií) z WGS 84 do S-JTSK."""
    return shapely.transform(geom, lambda xy: np.column_stack(transformer.transform(xy[:, 0], xy[:, 1])))

cr_m = do_krovaka(cr)
kraje_m = do_krovaka(kraje.geometry.to_numpy())     # pole 14 polygonů v metrech

kraje["rozloha_km2"] = (shapely.area(kraje_m) / 1e6).round(0)
kraje["obvod_km"] = (shapely.length(kraje_m) / 1e3).round(0)

print(f"Rozloha ČR: {cr_m.area / 1e6:,.0f} km² (oficiálně 78 871 km²)")
kraje[["name", "rozloha_km2", "obvod_km"]].sort_values("rozloha_km2", ascending=False)

Sjednocením (`union_all`) všech krajů by měla vzniknout celá republika. Ověříme to porovnáním s polygonem ČR – `symmetric_difference` vrátí plochu, ve které se obě geometrie liší:

In [ ]:
kraje_dohromady = shapely.union_all(kraje_m)
rozdil = kraje_dohromady.symmetric_difference(cr_m)

print("Typ sjednocení:", kraje_dohromady.geom_type)
print(f"Plocha, ve které se liší od polygonu ČR: {rozdil.area / 1e6:.2f} km²")

## 3. Které kraje spolu sousedí?

Prostorové vztahy testují **predikáty** – funkce vracející `True`/`False`:

| Funkce | Otázka |
|---|---|
| `intersects` | mají něco společného? |
| `touches` | dotýkají se jen hranicí? |
| `contains` / `within` | leží jedno celé uvnitř druhého? |
| `crosses` | kříží se (např. řeka hranici kraje)? |

Vektorizovaná funkce `shapely.touches` porovná všech 14 krajů se všemi najednou (vznikne tabulka 14 × 14):

In [ ]:
k = kraje_m
sousedi = shapely.touches(k[:, np.newaxis], k[np.newaxis, :])

kraje["pocet_sousedu"] = sousedi.sum(axis=1)
kraje["sousedi"] = [", ".join(kraje["name"][radek]) for radek in sousedi]
kraje[["name", "pocet_sousedu", "sousedi"]].sort_values("pocet_sousedu", ascending=False)

Zajímavý je vztah Prahy a Středočeského kraje. Praha leží „uprostřed“ Středočeského kraje, ale protože vyplňuje jeho **díru**, Shapely říká, že v něm neleží – jen se ho dotýká:

In [ ]:
praha_kraj = kraje_m[kraje["name"] == "Praha"][0]
stredocesky_m = kraje_m[kraje["name"] == "Středočeský"][0]

print("Praha leží ve Středočeském kraji (within):", praha_kraj.within(stredocesky_m))
print("Praha se dotýká Středočeského kraje:      ", praha_kraj.touches(stredocesky_m))
print("Praha leží v obalu Středoč. kraje bez díry:", praha_kraj.within(Polygon(stredocesky_m.exterior)))

## 4. Města a kraje

Sídla máme pro celý svět (~7 300 bodů). Města v ČR vybereme predikátem `contains` – vektorizovaná funkce otestuje všechna sídla jedním voláním, bez cyklu:

In [ ]:
body = mesta_svet.geometry.to_numpy()
maska = shapely.contains(cr, body)          # pole True/False pro všech ~7 300 sídel
print("Sídel v ČR:", maska.sum())

Ke každému městu teď najdeme kraj, ve kterém leží – opět jedním vektorizovaným voláním (12 měst × 14 krajů):

In [ ]:
mesta = mesta_svet.loc[maska, ["name", "pop_max", "geometry"]].reset_index(drop=True)
mesta["name"] = mesta["name"].replace({"Prague": "Praha", "Pizen": "Plzeň", "Usti Nad Labem": "Ústí nad Labem"})
mesta_m = do_krovaka(mesta.geometry.to_numpy())

v_kraji = shapely.contains(kraje_m[:, np.newaxis], mesta_m[np.newaxis, :])   # tabulka kraje × města
mesta["kraj"] = kraje["name"].to_numpy()[v_kraji.argmax(axis=0)]
mesta[["name", "pop_max", "kraj"]].sort_values("pop_max", ascending=False)

> V datech Natural Earth chybí Karlovy Vary – vrstva obsahuje jen nejvýznamnější sídla.

## 5. Jak daleko jsou města od státní hranice?

Hranice státu je `boundary` polygonu (linie). `distance` vrátí nejkratší vzdálenost, `nearest_points` dvojici nejbližších bodů – nejbližší bod na hranici použijeme pro vykreslení spojnice.

In [ ]:
hranice = cr_m.boundary
mesta["od_hranice_km"] = (shapely.distance(mesta_m, hranice) / 1000).round(1)

fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr_m, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")
for nazev, mesto in zip(mesta["name"], mesta_m):
    bod_na_hranici = nearest_points(mesto, hranice)[1]
    plot_line(LineString([mesto, bod_na_hranici]), ax=ax, color="tab:red", linewidth=1, add_points=False)
    plot_points(mesto, ax=ax, color="black", markersize=4)
    ax.annotate(nazev, (mesto.x, mesto.y), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Nejkratší spojnice měst se státní hranicí")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

mesta[["name", "od_hranice_km"]].sort_values("od_hranice_km")

Pomocí `buffer` a `difference` můžeme vymezit „vnitrozemí“ – území ČR, které je od hranice dál než 30 km:

In [ ]:
pohranici = hranice.buffer(30_000)              # pás 30 km na obě strany hranice
vnitrozemi = cr_m.difference(pohranici)         # ČR minus pás = vnitrozemí

print(f"Vnitrozemí tvoří {vnitrozemi.area / cr_m.area:.0%} rozlohy ČR.")
print("Města ve vnitrozemí:", ", ".join(mesta["name"][shapely.contains(vnitrozemi, mesta_m)]))

fig, ax = plt.subplots(figsize=(9, 5))
plot_polygon(cr_m, ax=ax, add_points=False, color="lightgrey", edgecolor="black")
plot_polygon(vnitrozemi, ax=ax, add_points=False, color="tab:orange", alpha=0.7)
plot_points(mesta_m, ax=ax, color="black", markersize=4)
ax.set_title("Vnitrozemí: více než 30 km od státní hranice")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 6. Řeky na území ČR

Z celosvětových řek vybereme ty, které zasahují do ČR, ořízneme je hranicí (`intersection`) a sloučíme úseky stejného jména.

In [ ]:
reky_geom = reky_svet.geometry.to_numpy()
v_cr = shapely.intersects(cr, reky_geom)
orez = shapely.intersection(reky_geom[v_cr], cr)

reky = (
    gpd.GeoDataFrame({"name": reky_svet["name"].to_numpy()[v_cr]}, geometry=orez, crs="EPSG:4326")
    .dissolve(by="name")            # sloučení úseků – GeoPandas uvnitř volá shapely.union_all
    .reset_index()
)
reky = reky[~reky.geometry.is_empty].reset_index(drop=True)
reky_m = do_krovaka(reky.geometry.to_numpy())
reky["delka_v_cr_km"] = (shapely.length(reky_m) / 1000).round(0)
reky[["name", "delka_v_cr_km"]].sort_values("delka_v_cr_km", ascending=False)

> Délky ovlivňuje zjednodušení dat: Vltava vyšla kratší než skutečných ~430 km, naopak pod jménem Svitava jsou v Natural Earth uloženy i úseky jiných toků. Shapely počítá správně – výsledek je ale vždy jen tak přesný jako vstupní data.

Kterými kraji Vltava protéká? Stačí predikát `intersects`:

In [ ]:
vltava_m = reky_m[reky["name"] == "Vltava"][0]
print("Vltava protéká kraji:", ", ".join(kraje["name"][shapely.intersects(vltava_m, kraje_m)]))

## 7. Která města leží u řeky?

Kolem všech řek vytvoříme pásmo 5 km (`buffer`), pásma sloučíme do jednoho polygonu a otestujeme, která města do něj padnou.

In [ ]:
pasmo = shapely.union_all(shapely.buffer(reky_m, 5_000)).intersection(cr_m)
u_reky = shapely.contains(pasmo, mesta_m)

print(f"Pásmo pokrývá {pasmo.area / cr_m.area:.0%} území ČR.")
print("Města u řeky:  ", ", ".join(mesta["name"][u_reky]))
print("Města mimo:    ", ", ".join(mesta["name"][~u_reky]))

fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr_m, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")
plot_polygon(pasmo, ax=ax, add_points=False, color="lightblue", alpha=0.7)
for geom in reky_m:
    plot_line(geom, ax=ax, color="tab:blue", linewidth=1, add_points=False)
plot_points(mesta_m[u_reky], ax=ax, color="tab:green", markersize=6)
plot_points(mesta_m[~u_reky], ax=ax, color="tab:red", markersize=6)
ax.set_title("Pásmo 5 km kolem řek (zeleně města u řeky, červeně mimo)")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 8. Spádové oblasti měst

**Voroného polygony** rozdělí území tak, že každé místo připadne **nejbližšímu** městu. Výsledek porovnáme se skutečnými hranicemi krajů.

In [ ]:
voronoi = shapely.voronoi_polygons(MultiPoint(list(mesta_m)), extend_to=cr_m)
oblasti = [cr_m.intersection(bunka) for bunka in voronoi.geoms]

fig, ax = plt.subplots(figsize=(11, 6))
barvy = plt.cm.tab20(np.linspace(0, 1, len(oblasti)))
for oblast, barva in zip(oblasti, barvy):
    plot_polygon(oblast, ax=ax, add_points=False, color=barva, alpha=0.6, edgecolor="none")
for kraj in kraje_m:
    plot_line(kraj.exterior, ax=ax, color="black", linewidth=0.8, add_points=False)
plot_points(mesta_m, ax=ax, color="black", markersize=4)
for nazev, m in zip(mesta["name"], mesta_m):
    ax.annotate(nazev, (m.x, m.y), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Spádové oblasti měst (barevně) a hranice krajů (černě)")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 9. Kam umístit popisek?

`centroid` je geometrické těžiště. U tvarů jako „podkova“ ale může ležet **mimo** polygon – typickým příkladem je Chorvatsko. Pro popisky v mapě je proto lepší `representative_point()`, který vždy leží uvnitř.

In [ ]:
chorvatsko = staty.loc[staty["adm0_a3"] == "HRV", "geometry"].iloc[0]

print("Leží centroid v Chorvatsku?          ", chorvatsko.contains(chorvatsko.centroid))
print("Leží representative_point v Chorvatsku?", chorvatsko.contains(chorvatsko.representative_point()))

fig, ax = plt.subplots(figsize=(6, 5))
bosna = staty.loc[staty["adm0_a3"] == "BIH", "geometry"].iloc[0]
plot_polygon(bosna, ax=ax, add_points=False, color="white", edgecolor="grey", linestyle=":")
ax.annotate("Bosna a Hercegovina", bosna.representative_point().coords[0], ha="center", fontsize=8, color="grey")
plot_polygon(chorvatsko, ax=ax, add_points=False, color="lightgrey", edgecolor="black")
plot_points(chorvatsko.centroid, ax=ax, color="red", markersize=10, label="centroid")
plot_points(chorvatsko.representative_point(), ax=ax, color="green", markersize=10, label="representative_point")
ax.legend(loc="lower left")
ax.set_title("Chorvatsko")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Stejnou funkci použijeme pro popisky krajů:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for kraj, barva in zip(kraje_m, plt.cm.Pastel1(np.linspace(0, 1, len(kraje_m)))):
    plot_polygon(kraj, ax=ax, add_points=False, color=barva, edgecolor="grey")
for nazev, bod in zip(kraje["name"], shapely.point_on_surface(kraje_m)):   # = representative_point
    ax.annotate(nazev, (bod.x, bod.y), ha="center", fontsize=8)
ax.set_title("Kraje ČR s popisky v reprezentativních bodech")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 10. Celý svět najednou: prostorový index STRtree

Úloha: ke **každému** ze ~7 300 sídel světa najít region, ve kterém leží (kraj, spolkový stát, provincii… – v datech je jich ~4 600). Naivně by to znamenalo testovat každé sídlo proti každému regionu, tedy přes 33 milionů testů. **STRtree** nejdřív podle obalových obdélníků rychle vybere jen pár kandidátů a přesný test udělá jen pro ně. Přesně tohle dělá uvnitř `geopandas.sjoin`.

In [ ]:
regiony_geom = regiony.geometry.to_numpy()

t0 = time.perf_counter()
strom = shapely.STRtree(regiony_geom)
idx_bod, idx_region = strom.query(body, predicate="within")     # páry indexů (sídlo, region)
t_strom = time.perf_counter() - t0

# naivní dvojitý cyklus – změříme na 50 sídlech a přepočteme na všechna
t0 = time.perf_counter()
for p in body[:50]:
    for region in regiony_geom:
        region.contains(p)
t_naivne = (time.perf_counter() - t0) / 50 * len(body)

print(f"STRtree (včetně stavby stromu): {t_strom:.2f} s | naivně (odhad): {t_naivne:.1f} s → cca {t_naivne / t_strom:.0f}× rychleji")
print(f"Přiřazeno {len(idx_bod)} z {len(body)} sídel.")

# ukázka výsledku
pd.DataFrame({
    "sídlo": mesta_svet["name"].to_numpy()[idx_bod],
    "region": regiony["name"].to_numpy()[idx_region],
    "stát": regiony["admin"].to_numpy()[idx_region],
}).sample(8, random_state=1)

In [ ]:
bez_statu = np.setdiff1d(np.arange(len(body)), idx_bod)
mesta_svet.iloc[bez_statu][["name", "adm0name"]].head(8)

Nepřiřazená sídla jsou přímořská a ostrovní města, jejichž bod leží těsně za zjednodušenou pobřežní linií. Při spojování dat různé přesnosti se proto hodí tolerance, např. `strom.query_nearest(body, max_distance=…)`.

## 11. Zjednodušení hranice

`simplify` (algoritmus Douglas–Peucker) odstraní vrcholy, které nejsou pro tvar podstatné – hodí se pro webové mapy nebo přehledky.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, tol in zip(axes, [0, 1_000, 5_000, 20_000]):
    zj = cr_m.simplify(tol) if tol else cr_m
    plot_polygon(zj, ax=ax, add_points=False, color="tab:purple", alpha=0.5)
    ax.set_title(f"tolerance {tol / 1000:.0f} km\n{shapely.get_num_coordinates(zj)} vrcholů, "
                 f"{zj.area / 1e6:,.0f} km²", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

## 12. Limity

**Rovinné výpočty.** Shapely nezná souřadnicové systémy ani kulatost Země – vše počítá v rovině. Proto jsme data převáděli do S-JTSK. Pro výpočty přes celý svět je potřeba knihovna pyproj (`pyproj.Geod`) nebo jiné nástroje.

**Nevalidní geometrie.** Simulujme chybu digitalizace: v hranici Vysočiny prohodíme dva sousední vrcholy. Na mapě rozdíl nepoznáte, ale hranice teď sama sebe protíná:

In [ ]:
vysocina = kraje_m[kraje["name"] == "Vysočina"][0]
jihomoravsky = kraje_m[kraje["name"] == "Jihomoravský"][0]

souradnice = list(vysocina.exterior.coords)
souradnice[1], souradnice[2] = souradnice[2], souradnice[1]
poskozena = Polygon(souradnice)

print("Validní?", poskozena.is_valid, "|", shapely.is_valid_reason(poskozena))
try:
    poskozena.union(jihomoravsky)
except shapely.errors.GEOSException as chyba:
    print("Sjednocení s Jihomoravským krajem selhalo:", chyba)

opravena = shapely.make_valid(poskozena)
print("Po make_valid:", opravena.is_valid, "| sjednocení:", opravena.union(jihomoravsky).geom_type)

**Přesnost čísel.** Souřadnice jsou desetinná čísla, takže i bod spočítaný přímo na Vltavě na ní podle Shapely „neleží“ – je od ní vzdálen o nepatrný zlomek metru. Řešením je tolerance (`dwithin`):

In [ ]:
bod_na_vltave = vltava_m.interpolate(0.5, normalized=True)    # bod v polovině délky Vltavy

print("Leží bod na Vltavě?        ", vltava_m.intersects(bod_na_vltave))
print("Vzdálenost od Vltavy (m):  ", vltava_m.distance(bod_na_vltave))
print("Je do 1 mm od Vltavy?      ", shapely.dwithin(vltava_m, bod_na_vltave, 0.001))

**Další omezení:** výpočty probíhají jen ve 2D (výška se ignoruje) a Shapely neumí číst soubory ani pracovat s atributy – proto se používá spolu s GeoPandas.

## Zdroje

[Dokumentace Shapely](https://shapely.readthedocs.io/) · [GEOS](https://libgeos.org/) · [GeoPandas](https://geopandas.org/) · [pyproj](https://pyproj4.github.io/pyproj/) · [Natural Earth](https://www.naturalearthdata.com/)